# 06: RQ2 results, conditions A to E and the core-only ablation

RQ2 asks whether a reranker can learn what makes a paper inspiring. The conditions change one thing at a time:

| Cond. | What | Compared with | Question it answers |
|---|---|---|---|
| A | BM25, top 20 | | floor |
| B | frozen Qwen3-Embedding-4B, top 20 (its top 100 is the shortlist) | A | keywords vs meaning |
| C | Qwen3-Reranker-0.6B off the shelf, rescoring B's top 100 | B | does a careful second look help? |
| D | C plus LoRA trained on (question, credited, rejected) pairs | C | **does learning from authors help?** |
| E | D plus mined negatives (unlabeled shortlist papers at ranks 30 to 100, weight 0.3) | D | does more but noisier data help? |
| D_core | D trained on core questions only | D | do subfield questions add useful training signal? |

This notebook reads whatever result files exist in `results/` and says which are missing. Metrics are computed here from the run files with the same code for every condition (`insight.rerank.per_question`), core and subfield separately. Pairwise accuracy needs scores for every credited and rejected paper, so it exists only for C, D, E and D_core (their `pairs_*.parquet`). Confidence intervals (paired bootstrap over projects) come with `insight.stats` in the Mon Nov 30 item; here we show mean per-question differences only.

In [1]:
import numpy as np, pandas as pd, yaml
from insight import data, rerank as R, train_reranker as T
pd.set_option("display.precision", 4)
ds = data.load(data.DataConfig.from_yaml(data.ROOT / "configs" / "data.yaml"))
table = ds.table
qids = table.index.tolist()
res = data.ROOT / "results"
RUNS = {"A": "run_A.parquet", "B": "shortlist_B.parquet", "C": "run_C.parquet", "D": "run_D.parquet",
        "E": "run_E.parquet", "D_core": "run_D_core.parquet"}
present = {}
for cond, f in RUNS.items():
    ok = (res / f).exists()
    print(f"{cond:7s} {f:22s} {'found' if ok else 'not in results/ yet'}")
    if ok:
        run = pd.read_parquet(res / f)
        pp = res / f"pairs_{cond}.parquet"
        pairs = pd.read_parquet(pp) if pp.exists() else pd.DataFrame(columns=["query_id", "doc_id", "label", "score"])
        present[cond] = R.per_question(run, pairs, table, qids)

A       run_A.parquet          found


B       shortlist_B.parquet    not in results/ yet
C       run_C.parquet          not in results/ yet
D       run_D.parquet          not in results/ yet
E       run_E.parquet          not in results/ yet
D_core  run_D_core.parquet     not in results/ yet


## 1. Metrics per condition

Means over questions (a question with no credited paper in the run scores 0, as everywhere in this repo). The paper reports core and subfield questions separately, so we do too.

In [2]:
cols = ["recall@5", "recall@20", "recall@100", "ndcg@20", "mrr", "pairwise_acc"]
rows = []
for cond, pq in present.items():
    for kind, g in pq.groupby("kind"):
        rows.append({"condition": cond, "kind": kind, "n": len(g), **g[cols].mean().to_dict()})
summary = pd.DataFrame(rows).set_index(["kind", "condition"]).sort_index(level=0, sort_remaining=False)
summary

,,n,recall@5,recall@20,recall@100,ndcg@20,mrr,pairwise_acc
kind,condition,,,,,,,
core,A,207,0.1225,0.2406,0.4171,0.1644,0.2349,NaN
subfield,A,687,0.1270,0.2853,0.4876,0.2275,0.3536,NaN


## 2. The comparisons, one change at a time

For each neighbouring pair we take the per-question difference (second minus first) on the same questions and report its mean, plus how many questions got better, worse or stayed the same. A comparison prints "waiting" until both files exist.

In [3]:
COMPARISONS = [("A", "B", "keywords vs meaning"), ("B", "C", "untrained second look"),
               ("C", "D", "learning from authors (key result)"), ("D", "E", "adding mined negatives"),
               ("D_core", "D", "adding subfield questions to training")]

def compare(a, b, metric):
    x, y = present[a], present[b]
    d = (y[metric] - x[metric]).dropna()
    out = []
    for kind in ["core", "subfield"]:
        dk = d[x.kind.reindex(d.index) == kind]
        out.append({"kind": kind, "n": len(dk), "mean diff": dk.mean(), "better": int((dk > 0).sum()),
                    "worse": int((dk < 0).sum()), "same": int((dk == 0).sum())})
    return pd.DataFrame(out).set_index("kind")

for a, b, what in COMPARISONS:
    if a in present and b in present:
        for metric in ["recall@20", "ndcg@20"] + (["pairwise_acc"] if present[a].pairwise_acc.notna().any() else []):
            print(f"\n{a} -> {b} ({what}), {metric}:")
            print(compare(a, b, metric).round(4).to_string())
    else:
        missing = [c for c in (a, b) if c not in present]
        print(f"{a} -> {b} ({what}): waiting for {', '.join(missing)}")

A -> B (keywords vs meaning): waiting for B
B -> C (untrained second look): waiting for B, C
C -> D (learning from authors (key result)): waiting for C, D
D -> E (adding mined negatives): waiting for D, E
D_core -> D (adding subfield questions to training): waiting for D_core, D


## 3. What condition E adds to D's training data

Condition E keeps every author pair of D (weight 1) and adds, for each training question, up to `mine_per_query` papers drawn from ranks 30 to 100 of its shortlist that the authors never labeled. Each mined paper is paired with every credited paper of the question at weight 0.3. Papers credited for *any* question of the same project are never mined (subfield questions share credited papers with their core question). Settings come from `configs/E.yaml`.

**Stand-in:** condition E mines from B's shortlist. Until `results/shortlist_B.parquet` exists, the cell below measures the same quantities on BM25's top 100 (`run_A.parquet`), so the counts show the size of the data, not the final values. It prints which shortlist it used.

In [4]:
ecfg = T.TrainConfig.from_yaml(data.ROOT / "configs" / "E.yaml")
sl_name = "shortlist_B.parquet" if (res / "shortlist_B.parquet").exists() else "run_A.parquet"
sl = pd.read_parquet(res / sl_name)
print(f"shortlist mined from: {sl_name}; mine_ranks {ecfg.mine_ranks}, weight {ecfg.mine_weight}, "
      f"per question {ecfg.mine_per_query}, exclude project positives {ecfg.mine_exclude_project_positives}")
splits = T.splits_from_csv(res / "folds.csv")
rows = []
for sp in splits:
    pairs, val = T.fold_pairs(table, sp, ecfg, sl)
    rows.append({"fold": sp.fold, "author pairs": int((pairs.source == "author").sum()),
                 "mined pairs": int((pairs.source == "mined").sum()),
                 "mined papers": int(pairs[pairs.source == "mined"][["query_id", "neg_id"]].drop_duplicates().shape[0]),
                 "train questions": len([q for q in sp.train if table.at[q, "kind"] in ecfg.kinds])})
fold_tab = pd.DataFrame(rows).set_index("fold")
fold_tab["mined share of weight"] = (fold_tab["mined pairs"] * ecfg.mine_weight) / (
    fold_tab["mined pairs"] * ecfg.mine_weight + fold_tab["author pairs"])
fold_tab

shortlist mined from: run_A.parquet; mine_ranks [30, 100], weight 0.3, per question 10, exclude project positives True


,author pairs,mined pairs,mined papers,train questions,mined share of weight
fold,,,,,
0,17720,30090,5680,568,0.3375
1,17661,31200,5770,577,0.3464
2,17333,31080,5690,569,0.3498
3,17848,30510,5680,568,0.3390
4,18010,31540,5730,573,0.3444


### How noisy is the mining band?

Over all 894 questions (no training is done here, so using every question is fine for a description), we look at the shortlist papers at ranks 30 to 100 and count how many are credited or rejected for that question, how many are credited for another question of the same project, and how many are unlabeled (the pool E samples from). The credited share inside the band is a reminder that the band does contain inspirations: the unlabeled ones are not known to be negatives.

In [5]:
lo, hi = ecfg.mine_ranks
band = sl[sl["rank"].between(lo, hi)].copy()
proj_pos = {pid: {d for ps in g.positives for d in ps} for pid, g in table.groupby("paper_id")}
def status(q, d):
    r = table.loc[q]
    if d in r.positives: return "credited for this question"
    if d in r.negatives: return "rejected for this question"
    if d in proj_pos[r.paper_id]: return "credited for another question of the project"
    return "unlabeled (mining pool)"
band["status"] = [status(q, d) for q, d in zip(band.query_id, band.doc_id)]
band["kind"] = table.kind.reindex(band.query_id).values
counts = band.groupby(["kind", "status"]).size().unstack(0).fillna(0).astype(int)
counts.loc["total"] = counts.sum()
print(f"shortlist: {sl_name}, ranks {lo} to {hi}")
display(counts)
display((counts.drop(index="total") / counts.loc["total"]).round(4))

shortlist: run_A.parquet, ranks 30 to 100


kind,core,subfield
status,,
credited for another question of the project,280,441
credited for this question,84,661
rejected for this question,342,809
unlabeled (mining pool),13991,46866
total,14697,48777


kind,core,subfield
status,,
credited for another question of the project,0.0191,0.0090
credited for this question,0.0057,0.0136
rejected for this question,0.0233,0.0166
unlabeled (mining pool),0.9520,0.9608


## 4. Status

The tables above use only the result files that exist. When C, D, E and D_core land (H100 queue items 3 to 6), rerun this notebook; nothing in it needs editing. Reading the comparisons:

- **C to D** is the key result: same reranker, same instruction, same shortlist, only training differs.
- **D to E** tells whether mined negatives help despite their noise.
- **D_core to D** tells whether subfield questions (more pairs, but their positives overlap with the core question's) add signal.